# Lesson 5a — House price predictor, executable line by line

Runnable version of the [Lesson 5a walkthrough](../05a_walkthrough.md).

This sits between `pragma_mini.py` (3 keys) and the streaming churn predictor
(4 keys × 15 events). One record per house, **10 attributes**, **5-class**
price classification.

What's new here:
- Realistic **inter-attribute correlations** (beach houses tend to have pools, etc.)
- 5-class prediction (not binary)
- A nuanced pre-train vs baseline comparison

## 🧰 Lesson reference legend

- **L1** — the 5-line training loop
- **L1b** — architecture vs training
- **L1c** — gradient descent details
- **L2** — tokens & embeddings
- **L3** — attention
- **L4** — masked language modelling
- **L5** — pragma_mini.py


## 0 — Imports and seeds

In [ ]:
import random
import copy
import torch
import torch.nn as nn

torch.manual_seed(0)
random.seed(0)

## 1 — Vocabulary (L2 + L5)

10 keys per house. Each key has 2-5 categorical values. Total vocab: ~48 tokens.


In [ ]:
KEYS = ["bedrooms", "bathrooms", "size", "age", "neighborhood",
        "garage", "pool", "garden", "schools", "condition"]

VALUE_BUCKETS = {
    "bedrooms":     ["1bed", "2bed", "3bed", "4bed", "5+bed"],
    "bathrooms":    ["1bath", "2bath", "3+bath"],
    "size":         ["small", "medium", "large", "huge"],
    "age":          ["new", "modern", "older", "vintage"],
    "neighborhood": ["downtown", "suburb", "rural", "beach"],
    "garage":       ["nogarage", "1car", "2car"],
    "pool":         ["nopool", "haspool"],
    "garden":       ["nogarden", "smallgarden", "largegarden"],
    "schools":      ["poorschool", "avgschool", "goodschool", "excschool"],
    "condition":    ["poorcond", "faircond", "goodcond", "exccond"],
}

PAD, MASK = "<pad>", "<mask>"
VALUES = [v for vs in VALUE_BUCKETS.values() for v in vs]
vocab  = [PAD, MASK] + KEYS + VALUES
tok2id = {t: i for i, t in enumerate(vocab)}
V      = len(vocab)

PRICE_CLASSES = ["bargain", "cheap", "average", "expensive", "luxury"]
print(f"vocab size: {V}")
print(f"vocab: {vocab}")

## 2 — Realistic correlations (the secret sauce)

Without correlations, MLM has nothing context-dependent to learn — every value would be predictable from its key alone. With correlations, MLM can learn that beach houses usually have pools, vintage houses are usually smaller, etc.

We define 5 "house archetypes" with weighted attribute distributions.


In [ ]:
ARCHETYPES = {
    "rural_cottage": {
        "bedrooms":     {"1bed":2, "2bed":5, "3bed":3, "4bed":1, "5+bed":0},
        "bathrooms":    {"1bath":6, "2bath":3, "3+bath":1},
        "size":         {"small":5, "medium":4, "large":1, "huge":0},
        "age":          {"new":0, "modern":1, "older":4, "vintage":5},
        "neighborhood": {"downtown":0, "suburb":1, "rural":8, "beach":1},
        "garage":       {"nogarage":3, "1car":5, "2car":2},
        "pool":         {"nopool":9, "haspool":1},
        "garden":       {"nogarden":1, "smallgarden":3, "largegarden":6},
        "schools":      {"poorschool":4, "avgschool":4, "goodschool":2, "excschool":0},
        "condition":    {"poorcond":2, "faircond":4, "goodcond":3, "exccond":1},
    },
    "suburban_family": {
        "bedrooms":     {"1bed":0, "2bed":1, "3bed":5, "4bed":3, "5+bed":1},
        "bathrooms":    {"1bath":1, "2bath":6, "3+bath":3},
        "size":         {"small":1, "medium":5, "large":3, "huge":1},
        "age":          {"new":2, "modern":5, "older":2, "vintage":1},
        "neighborhood": {"downtown":1, "suburb":7, "rural":1, "beach":1},
        "garage":       {"nogarage":1, "1car":3, "2car":6},
        "pool":         {"nopool":7, "haspool":3},
        "garden":       {"nogarden":1, "smallgarden":5, "largegarden":4},
        "schools":      {"poorschool":0, "avgschool":3, "goodschool":5, "excschool":2},
        "condition":    {"poorcond":0, "faircond":2, "goodcond":5, "exccond":3},
    },
    "urban_apartment": {
        "bedrooms":     {"1bed":4, "2bed":4, "3bed":2, "4bed":0, "5+bed":0},
        "bathrooms":    {"1bath":5, "2bath":4, "3+bath":1},
        "size":         {"small":7, "medium":2, "large":1, "huge":0},
        "age":          {"new":2, "modern":5, "older":2, "vintage":1},
        "neighborhood": {"downtown":9, "suburb":1, "rural":0, "beach":0},
        "garage":       {"nogarage":7, "1car":2, "2car":1},
        "pool":         {"nopool":9, "haspool":1},
        "garden":       {"nogarden":8, "smallgarden":2, "largegarden":0},
        "schools":      {"poorschool":1, "avgschool":3, "goodschool":4, "excschool":2},
        "condition":    {"poorcond":1, "faircond":3, "goodcond":4, "exccond":2},
    },
    "luxury_beach": {
        "bedrooms":     {"1bed":0, "2bed":1, "3bed":2, "4bed":4, "5+bed":3},
        "bathrooms":    {"1bath":0, "2bath":2, "3+bath":8},
        "size":         {"small":0, "medium":1, "large":4, "huge":5},
        "age":          {"new":5, "modern":4, "older":1, "vintage":0},
        "neighborhood": {"downtown":0, "suburb":0, "rural":0, "beach":10},
        "garage":       {"nogarage":0, "1car":1, "2car":9},
        "pool":         {"nopool":1, "haspool":9},
        "garden":       {"nogarden":0, "smallgarden":2, "largegarden":8},
        "schools":      {"poorschool":0, "avgschool":1, "goodschool":3, "excschool":6},
        "condition":    {"poorcond":0, "faircond":0, "goodcond":3, "exccond":7},
    },
    "old_townhouse": {
        "bedrooms":     {"1bed":1, "2bed":4, "3bed":4, "4bed":1, "5+bed":0},
        "bathrooms":    {"1bath":4, "2bath":5, "3+bath":1},
        "size":         {"small":2, "medium":5, "large":2, "huge":1},
        "age":          {"new":0, "modern":1, "older":4, "vintage":5},
        "neighborhood": {"downtown":5, "suburb":4, "rural":1, "beach":0},
        "garage":       {"nogarage":4, "1car":4, "2car":2},
        "pool":         {"nopool":9, "haspool":1},
        "garden":       {"nogarden":4, "smallgarden":5, "largegarden":1},
        "schools":      {"poorschool":2, "avgschool":4, "goodschool":3, "excschool":1},
        "condition":    {"poorcond":3, "faircond":4, "goodcond":2, "exccond":1},
    },
}

ARCHETYPE_WEIGHTS = {
    "rural_cottage": 2, "suburban_family": 4, "urban_apartment": 2,
    "luxury_beach": 1, "old_townhouse": 2,
}

PRICE_WEIGHTS = {
    "bedrooms":     {"1bed":0,"2bed":30,"3bed":60,"4bed":90,"5+bed":120},
    "bathrooms":    {"1bath":0,"2bath":25,"3+bath":55},
    "size":         {"small":0,"medium":80,"large":160,"huge":260},
    "age":          {"new":60,"modern":30,"older":0,"vintage":20},
    "neighborhood": {"downtown":150,"suburb":50,"rural":0,"beach":200},
    "garage":       {"nogarage":0,"1car":20,"2car":50},
    "pool":         {"nopool":0,"haspool":30},
    "garden":       {"nogarden":0,"smallgarden":15,"largegarden":40},
    "schools":      {"poorschool":0,"avgschool":40,"goodschool":90,"excschool":150},
    "condition":    {"poorcond":-50,"faircond":0,"goodcond":50,"exccond":110},
}
PRICE_BASE = 100

def weighted_choice(weights_dict):
    items, weights = zip(*weights_dict.items())
    return random.choices(items, weights=weights, k=1)[0]

def random_house():
    arch = weighted_choice(ARCHETYPE_WEIGHTS)
    dists = ARCHETYPES[arch]
    return {k: weighted_choice(dists[k]) for k in KEYS}

def house_price(house):
    p = PRICE_BASE
    for k, v in house.items():
        p += PRICE_WEIGHTS[k][v]
    if house["neighborhood"] == "beach" and house["pool"] == "haspool": p += 80
    if house["age"] == "vintage" and house["condition"] == "poorcond":  p -= 60
    if house["size"] == "huge" and house["bedrooms"] == "5+bed":        p += 60
    p += random.gauss(0, 30)
    return max(0, p)

def price_to_class(price):
    if price < 250:   return 0
    if price < 450:   return 1
    if price < 700:   return 2
    if price < 1000:  return 3
    return 4

# Show a few sample houses
for archetype in ARCHETYPES:
    sample_dists = ARCHETYPES[archetype]
    sample = {k: weighted_choice(sample_dists[k]) for k in KEYS}
    p = house_price(sample)
    print(f"\n{archetype}:  ${p:.0f}k  ({PRICE_CLASSES[price_to_class(p)]})")
    for k in KEYS:
        print(f"  {k:14s} {sample[k]}")

## 3 — Build the dataset

8000 houses. Each becomes a flat 20-token sequence.


In [ ]:
def encode_house(house):
    ids = []
    for k in KEYS:
        ids.append(tok2id[k])
        ids.append(tok2id[house[k]])
    return ids

N_HOUSES = 8000

houses, classes = [], []
for _ in range(N_HOUSES):
    h = random_house()
    p = house_price(h)
    houses.append(h)
    classes.append(price_to_class(p))

X = torch.tensor([encode_house(h) for h in houses], dtype=torch.long)
y = torch.tensor(classes, dtype=torch.long)

print(f"shape X: {tuple(X.shape)}  ({X.size(1)} tokens per house)")
class_counts = [int((y == c).sum()) for c in range(5)]
for c, n in enumerate(class_counts):
    print(f"  {PRICE_CLASSES[c]:>10s}: {n} houses")

## 4 — Architecture (L1b + L2 + L3)

Identical to pragma_mini.py and the streaming churn example. The encoder is reusable across tasks — only the head changes.


In [ ]:
D_MODEL, N_HEADS, N_LAYERS = 32, 2, 2

class Encoder(nn.Module):
    def __init__(self, V, d=D_MODEL, heads=N_HEADS, layers=N_LAYERS, max_len=64):
        super().__init__()
        self.emb = nn.Embedding(V, d)
        self.pos = nn.Embedding(max_len, d)
        layer    = nn.TransformerEncoderLayer(d, heads, d*2, batch_first=True)
        self.enc = nn.TransformerEncoder(layer, layers)
    def forward(self, x):
        pos = torch.arange(x.size(1))
        return self.enc(self.emb(x) + self.pos(pos))

class MLMHead(nn.Module):
    def __init__(self, V, d=D_MODEL):
        super().__init__()
        self.proj = nn.Linear(d, V)
    def forward(self, h):
        return self.proj(h)

class PriceHead(nn.Module):
    def __init__(self, d=D_MODEL, n_classes=5):
        super().__init__()
        self.proj = nn.Linear(d, n_classes)
    def forward(self, h):
        pooled = h.mean(dim=1)
        return self.proj(pooled)

print(f"Encoder knobs: {sum(p.numel() for p in Encoder(V).parameters()):,}")
print(f"MLMHead knobs: {sum(p.numel() for p in MLMHead(V).parameters()):,}")
print(f"PriceHead knobs: {sum(p.numel() for p in PriceHead().parameters())}")

## 5 — Pre-train via masked language modelling (L4)

30% mask rate (higher than other lessons) so the model is forced to use context heavily to fill in missing values. That's what teaches it the inter-attribute correlations.


In [ ]:
KEY_IDS = torch.tensor([tok2id[k] for k in KEYS])

def mlm_mask(X_batch, p=0.30):
    X = X_batch.clone()
    y = torch.full_like(X, -100)
    is_value = ~torch.isin(X, KEY_IDS)
    pick = (torch.rand_like(X, dtype=torch.float) < p) & is_value
    y[pick] = X[pick]
    X[pick] = tok2id[MASK]
    return X, y

encoder   = Encoder(V)
mlm_head  = MLMHead(V)
opt       = torch.optim.AdamW(list(encoder.parameters()) + list(mlm_head.parameters()), lr=3e-3)
loss_fn   = nn.CrossEntropyLoss(ignore_index=-100)

print("Pre-training encoder via MLM for 3000 steps...")
for step in range(3000):
    idx       = torch.randint(0, N_HOUSES, (128,))
    xb, yb    = mlm_mask(X[idx])
    h         = encoder(xb)
    logits    = mlm_head(h)
    loss      = loss_fn(logits.reshape(-1, V), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 500 == 0:
        print(f"  step {step:4d}   MLM loss {loss.item():.3f}")

pretrained_encoder = copy.deepcopy(encoder)
print("\nDone. Cached as `pretrained_encoder`.")

## 6 — Downstream comparison

Two modes:
- **Frozen pre-trained encoder + linear price head** (foundation-model recipe)
- **Random-init encoder, end-to-end** (baseline, no pre-training)

Tried at 4 label counts: 50, 100, 500, 4000.


In [ ]:
perm  = torch.randperm(N_HOUSES)
split = int(N_HOUSES * 0.8)
tr_idx, te_idx = perm[:split], perm[split:]
X_te, y_te     = X[te_idx], y[te_idx]
print(f"Test set: {len(te_idx)} houses\n")

def freeze(mod):
    for p in mod.parameters(): p.requires_grad = False
    mod.eval()

def train_classifier(encoder, X_tr, y_tr, epochs=500, freeze_encoder=True, batch_size=128):
    head = PriceHead()
    if freeze_encoder:
        freeze(encoder)
        params = list(head.parameters())
    else:
        params = list(encoder.parameters()) + list(head.parameters())
    opt = torch.optim.AdamW(params, lr=3e-3)
    loss_fn = nn.CrossEntropyLoss()
    n = X_tr.size(0)
    for _ in range(epochs):
        if n > batch_size:
            idx = torch.randperm(n)[:batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
        else:
            xb, yb = X_tr, y_tr
        h      = encoder(xb)
        logits = head(h)
        loss   = loss_fn(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()
    head.eval()
    with torch.no_grad():
        logits = head(encoder(X_te))
        pred = logits.argmax(-1)
        acc  = (pred == y_te).float().mean().item()
        per_class = []
        for c in range(5):
            mask = (y_te == c)
            if mask.sum() == 0:
                per_class.append(float("nan"))
            else:
                per_class.append((pred[mask] == c).float().mean().item())
        valid = [r for r in per_class if r == r]
        macro_recall = sum(valid) / len(valid) if valid else float("nan")
    return acc, macro_recall, head

print(f"{'labels':>7} | {'pretrained acc':>14}  {'pretrained recall':>17} | "
      f"{'baseline acc':>12}  {'baseline recall':>15}")
print("-" * 84)
for n_labels in [50, 100, 500, 4000]:
    sub = tr_idx[:n_labels]
    X_tr, y_tr = X[sub], y[sub]
    enc_a = copy.deepcopy(pretrained_encoder)
    acc_a, rec_a, _ = train_classifier(enc_a, X_tr, y_tr, freeze_encoder=True)
    torch.manual_seed(n_labels)
    enc_b = Encoder(V)
    acc_b, rec_b, _ = train_classifier(enc_b, X_tr, y_tr, freeze_encoder=False)
    print(f"{n_labels:>7} | {acc_a:>14.3f}  {rec_a:>17.3f} | {acc_b:>12.3f}  {rec_b:>15.3f}")

## 7 — Sample predictions

Hand-craft a few houses and see what the model predicts.


In [ ]:
enc_final = copy.deepcopy(pretrained_encoder)
freeze(enc_final)
_, _, head_final = train_classifier(enc_final, X[tr_idx], y[tr_idx], epochs=1000, freeze_encoder=True)

demo_houses = [
    ("rural cottage", {
        "bedrooms": "2bed", "bathrooms": "1bath", "size": "small", "age": "vintage",
        "neighborhood": "rural", "garage": "1car", "pool": "nopool", "garden": "largegarden",
        "schools": "avgschool", "condition": "faircond"}),
    ("suburban family", {
        "bedrooms": "3bed", "bathrooms": "2bath", "size": "medium", "age": "modern",
        "neighborhood": "suburb", "garage": "2car", "pool": "nopool", "garden": "smallgarden",
        "schools": "goodschool", "condition": "goodcond"}),
    ("luxury beach", {
        "bedrooms": "5+bed", "bathrooms": "3+bath", "size": "huge", "age": "new",
        "neighborhood": "beach", "garage": "2car", "pool": "haspool", "garden": "largegarden",
        "schools": "excschool", "condition": "exccond"}),
]
for label, h in demo_houses:
    ids = torch.tensor([encode_house(h)])
    with torch.no_grad():
        logits = head_final(enc_final(ids))
        probs  = torch.softmax(logits, dim=-1)[0].tolist()
        pred   = logits.argmax(-1).item()
    bars = "  ".join(f"{PRICE_CLASSES[c]:>10s}={probs[c]*100:5.1f}%" for c in range(5))
    print(f"\n{label:18s}  prediction: {PRICE_CLASSES[pred]}")
    print(f"  {bars}")

## 8 — Things to try

1. **Drop the correlations.** Replace `random_house` with one that samples each attribute independently. Re-run pre-training. Does pre-training still help at 50 labels?

2. **Inspect the pre-trained embeddings.** Compare:
```python
pretrained_encoder.emb.weight[tok2id["beach"]]
pretrained_encoder.emb.weight[tok2id["haspool"]]
```
These should be more similar than `beach` and `nopool` after training — because beach houses tend to have pools.

3. **More layers.** Bump `N_LAYERS` to 4. Does pre-training help more or less?

## What's next

[Lesson 5b](../05b_walkthrough.md) — same recipe, applied to sequential event data (streaming sessions), where pre-training wins more decisively.
